# 2. Modelling — Iteration 4

Run after the Iteration 4 chunking notebook in the same project folder. GPU is required; there is no CPU fallback.

- Input chunk length (ICL): 365 days.
- Output chunk length (OCL): derived from the configured forecast dates (101 days by default).
- Training metrics: CSV and TensorBoard logs are written under the timestamped fit directory.
- The exact best checkpoint and matching artifacts are recorded for the prediction notebook.

Use the 2025 backtest mode in the chunking notebook to evaluate festive performance on unseen dates. The production validation window (May 23–August 31) is not a festive evaluation window.


In [ ]:
from pathlib import Path
import json, hashlib, uuid, platform
from datetime import datetime, timezone
import numpy as np
import pandas as pd


def digest(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(b)
    return h.hexdigest()


def write_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, allow_nan=False), encoding="utf-8")
    tmp.replace(path)


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def check_daily(frame, date_col, start, end, label):
    idx = pd.DatetimeIndex(pd.to_datetime(frame[date_col]))
    expected = pd.date_range(start, end, freq="D")
    if idx.has_duplicates or not idx.equals(expected):
        raise ValueError(
            f"{label}: duplicate, missing or unexpected dates; expected {len(expected)} daily rows, got {len(idx)}. Repair the source grid; missing observations are not assumed to be zero."
        )


SEGMENTS = {
    "SPLENDOR+": "100 CC",
    "HF DELUXE": "100 CC",
    "HF 100": "100 CC",
    "PASSION": "100 CC",
    "GLAMOUR": "125 CC",
    "SUPER SPLENDOR": "125 CC",
    "XTREME 125": "125 CC",
    "XPULSE": "PREMIUM",
    "XTREME 160": "PREMIUM",
    "XTREME 250": "PREMIUM",
    "DESTINI": "SCOOTER",
    "PLEASURE+": "SCOOTER",
    "XOOM": "SCOOTER",
}


In [ ]:
PROJECT_DIR = Path.cwd()
RUN_DIR_OVERRIDE = None
RUN_DIR = Path(
    RUN_DIR_OVERRIDE or read_json(PROJECT_DIR / "iteration3_active_run.json")["run_dir"]
)
if not (RUN_DIR / "DATA_READY").exists():
    raise RuntimeError("Data snapshot is incomplete")
cfg = read_json(RUN_DIR / "config.json")
dm = read_json(RUN_DIR / "data_manifest.json")
for name, expected in [
    ("config.json", dm["config_hash"]),
    ("calendar.parquet", dm["calendar_hash"]),
    ("selected_series.parquet", dm["membership_hash"]),
]:
    if digest(RUN_DIR / name) != expected:
        raise RuntimeError(f"Data snapshot changed: {name}; create a new run")
TIME, KEY, TARGET = (cfg["time_col"], cfg["group_col"], cfg["target_col"])
STATIC, FUTURE = (cfg["static_covariates"], cfg["future_covariates"])
ICL, OCL = (cfg["icl"], cfg["ocl"])
TRAIN_START, TRAIN_END = (
    pd.Timestamp(cfg["train_start"]),
    pd.Timestamp(cfg["train_end"]),
)
HISTORY_END = pd.Timestamp(cfg["val_end"])
FC_START, FC_END = (
    pd.Timestamp(cfg["forecast_start"]),
    pd.Timestamp(cfg["forecast_end"]),
)
VAL_OUTPUT_START = pd.Timestamp(cfg["val_output_start"])
VAL_INPUT_START = VAL_OUTPUT_START - pd.Timedelta(days=ICL)
assert HISTORY_END + pd.Timedelta(days=1) == FC_START
assert OCL == (FC_END - FC_START).days + 1
assert VAL_OUTPUT_START > TRAIN_END
print("Data snapshot:", RUN_DIR, "| Population:", cfg["population"])


In [ ]:
import torch, darts, inspect, pickle, gc
from darts import TimeSeries
from darts.models import TFTModel
from darts.utils.likelihood_models import NegativeBinomialLikelihood
import pytorch_lightning as pl

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not visible. Select your CUDA-enabled PyTorch environment/kernel. This notebook will not silently fall back to CPU."
    )
torch.set_float32_matmul_precision("high")
PRECISION = "bf16-mixed" if torch.cuda.is_bf16_supported() else "32-true"
print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
)
print(
    "Darts:",
    darts.__version__,
    "| PyTorch:",
    torch.__version__,
    "| precision:",
    PRECISION,
)


In [ ]:
from darts.dataprocessing.transformers import StaticCovariatesTransformer
from sklearn.preprocessing import OrdinalEncoder
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.loggers import CSVLogger, TensorBoardLogger

BATCH_SIZE = 64
ACCUMULATE_GRAD_BATCHES = 4
MAX_EPOCHS = 100
MAX_TRAIN_BATCHES = 12000
HIDDEN_SIZE, LSTM_LAYERS, HEADS = (32, 4, 4)
DROPOUT = 0.05
if "sample_weight" not in inspect.signature(TFTModel.fit).parameters:
    raise RuntimeError(
        "This Darts version lacks sample_weight. Use a compatible environment; do not drop weights."
    )
ATTEMPT = (
    datetime.now(timezone.utc).strftime("fit_%Y%m%d_%H%M%S_") + uuid.uuid4().hex[:8]
)
FIT_DIR = RUN_DIR / ATTEMPT
FIT_DIR.mkdir(exist_ok=False)
CACHE_DIR = FIT_DIR / "series_cache"
CACHE_DIR.mkdir()
MODEL_NAME = "daily_tft_" + ATTEMPT
WORK_DIR = FIT_DIR / "darts_logs"
WORK_DIR.mkdir()
CSV_LOGGER = CSVLogger(save_dir=str(FIT_DIR), name="training_metrics")
TENSORBOARD_LOGGER = TensorBoardLogger(save_dir=str(FIT_DIR), name="tensorboard")
print("TensorBoard log directory:", TENSORBOARD_LOGGER.log_dir)
print(f'''Launch with: tensorboard --logdir "{FIT_DIR / 'tensorboard'}"''')
calendar = pd.read_parquet(RUN_DIR / "calendar.parquet")
calendar[TIME] = pd.to_datetime(calendar[TIME])
check_daily(calendar, TIME, TRAIN_START, FC_END, "Calendar")
SHARED_COV = TimeSeries.from_dataframe(
    calendar, time_col=TIME, value_cols=FUTURE, freq="D", fill_missing_dates=False
).astype(np.float32)
SHARED_WEIGHT = TimeSeries.from_dataframe(
    calendar, time_col=TIME, value_cols=["WEIGHT"], freq="D", fill_missing_dates=False
).astype(np.float32)


In [ ]:
records, static_rows, seen, audit = ([], [], set(), [])
for entry in dm["chunks"]:
    path = RUN_DIR / entry["path"]
    if digest(path) != entry["sha256"]:
        raise RuntimeError(f"Chunk changed: {path}")
    frame = pd.read_parquet(path)
    frame[TIME] = pd.to_datetime(frame[TIME])
    for key, g in frame.groupby(KEY, sort=False):
        key = str(key)
        if key in seen:
            raise ValueError(f"Series appears in multiple chunks: {key}")
        seen.add(key)
        g = g.sort_values(TIME)
        expected_end = FC_END if cfg["mode"] == "festive_backtest_2025" else HISTORY_END
        check_daily(g, TIME, TRAIN_START, expected_end, key)
        if g[STATIC].isna().any().any() or (g[STATIC].nunique(dropna=False) > 1).any():
            raise ValueError(f"{key}: static attributes change or are missing")
        h = g.loc[g[TIME] <= HISTORY_END]
        values = h[TARGET].to_numpy(dtype=np.float32)
        if (
            not np.isfinite(values).all()
            or (values < 0).any()
            or (not np.equal(values, np.rint(values)).all())
        ):
            raise ValueError(f"{key}: invalid count target")
        train = h.loc[h[TIME] <= TRAIN_END, TARGET]
        if len(train) < ICL + OCL:
            raise ValueError(
                f"{key}: insufficient training history; needs a separate short-history treatment"
            )
        name = hashlib.sha256(key.encode("utf-8")).hexdigest() + ".npz"
        np.savez(
            CACHE_DIR / name,
            dates=h[TIME].to_numpy(dtype="datetime64[ns]"),
            sales=values,
        )
        records.append({"key": key, "file": name, "sha256": digest(CACHE_DIR / name)})
        static_rows.append(g[STATIC].iloc[0].astype(str).to_dict())
        audit.append(
            {
                KEY: key,
                "MODEL_FAMILY": str(g.MODEL_FAMILY.iloc[0]),
                "TRAIN_SALES": float(train.sum()),
                "NEVER_SOLD_IN_TRAIN": bool(train.sum() == 0),
                "HISTORY_SALES": float(values.sum()),
            }
        )
    del frame
    gc.collect()
    print("Validated/cached series:", len(records))
selected = pd.read_parquet(RUN_DIR / "selected_series.parquet")[KEY].astype(str)
if seen != set(selected):
    raise ValueError(
        f"Membership mismatch: {len(set(selected) - seen)} selected IDs lack source history"
    )
if not records:
    raise ValueError("No series")
static_df = pd.DataFrame(static_rows, columns=STATIC)
audit_df = pd.DataFrame(audit)
audit_df.to_csv(FIT_DIR / "series_coverage_audit.csv", index=False)
print(audit_df.groupby("MODEL_FAMILY")[["TRAIN_SALES", "NEVER_SOLD_IN_TRAIN"]].sum())
write_json(FIT_DIR / "cache_manifest.json", records)


In [ ]:
fit_size = int(static_df.nunique().max())
fit_frame = pd.DataFrame(
    {
        c: pd.Series(sorted(static_df[c].unique())).reindex(range(fit_size)).ffill()
        for c in STATIC
    }
)
dummy_dates = pd.date_range("2000-01-01", periods=2)
dummy = [
    TimeSeries.from_times_and_values(
        dummy_dates,
        np.zeros((2, 1), np.float32),
        columns=[TARGET],
        static_covariates=fit_frame.iloc[[i]].reset_index(drop=True),
    )
    for i in range(fit_size)
]
transformer = StaticCovariatesTransformer(
    transformer_cat=OrdinalEncoder(handle_unknown="error"), cols_cat=STATIC
)
transformer.fit(dummy)
encoded = []
for i in range(len(static_df)):
    ts = TimeSeries.from_times_and_values(
        dummy_dates,
        np.zeros((2, 1), np.float32),
        columns=[TARGET],
        static_covariates=static_df.iloc[[i]].reset_index(drop=True),
    )
    encoded.append(transformer.transform(ts).static_covariates.astype(np.float32))
counts = {c: int(static_df[c].nunique()) for c in STATIC}
embeddings = {c: (counts[c], min(50, (counts[c] + 1) // 2)) for c in STATIC}
for stat in encoded:
    for c in STATIC:
        v = float(stat[c].iloc[0])
        if not v.is_integer() or not 0 <= v < counts[c]:
            raise ValueError("Invalid embedding index")
static_df.insert(0, KEY, [r["key"] for r in records])
static_df.to_parquet(FIT_DIR / "static_raw.parquet", index=False)
with open(FIT_DIR / "static_transformer.pkl", "wb") as f:
    pickle.dump(transformer, f)
with open(FIT_DIR / "static_encoded.pkl", "wb") as f:
    pickle.dump(encoded, f)
SHARED_COV.to_pickle(FIT_DIR / "shared_cov.pkl")
del dummy, static_rows
gc.collect()


In [ ]:
from collections.abc import Sequence
from functools import lru_cache


class Targets(Sequence):

    def __init__(self, records, static_frames, split):
        self.records, self.static_frames, self.split = (records, static_frames, split)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self[j] for j in range(*i.indices(len(self)))]
        if i < 0:
            i += len(self)
        if not 0 <= i < len(self):
            raise IndexError(i)
        return self._get(i)

    @lru_cache(maxsize=128)
    def _get(self, i):
        with np.load(CACHE_DIR / self.records[i]["file"], allow_pickle=False) as z:
            dates = pd.DatetimeIndex(z["dates"])
            sales = z["sales"]
        if self.split == "train":
            mask = dates <= TRAIN_END
        elif self.split == "val":
            mask = (dates >= VAL_INPUT_START) & (dates <= HISTORY_END)
        elif self.split == "history":
            mask = (dates >= FC_START - pd.Timedelta(days=ICL)) & (dates <= HISTORY_END)
        else:
            raise ValueError(self.split)
        return TimeSeries.from_times_and_values(
            dates[mask],
            sales[mask, None],
            columns=[TARGET],
            static_covariates=self.static_frames[i],
        )


class Shared(Sequence):

    def __init__(self, ts, n):
        self.ts, self.n = (ts, n)

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self.ts for _ in range(*i.indices(self.n))]
        if i < 0:
            i += self.n
        if not 0 <= i < self.n:
            raise IndexError(i)
        return self.ts


In [ ]:
train_seq = Targets(records, encoded, "train")
val_seq = Targets(records, encoded, "val")
train_cov = Shared(SHARED_COV, len(records))
val_cov = Shared(SHARED_COV, len(records))
train_weights = Shared(SHARED_WEIGHT, len(records))
val_weights = Shared(SHARED_WEIGHT, len(records))
assert train_seq[0].end_time() == TRAIN_END
assert val_seq[0].start_time() == VAL_INPUT_START and len(val_seq[0]) == ICL + OCL
print("Validation outputs:", VAL_OUTPUT_START.date(), "to", HISTORY_END.date())
print(
    "Train weights:",
    np.unique(SHARED_WEIGHT.slice(TRAIN_START, TRAIN_END).values(), return_counts=True),
)
total_windows = len(records) * (len(train_seq[0]) - ICL - OCL + 1)
print("Available training windows:", f"{total_windows:,}")
print(
    "Maximum windows per partial epoch:",
    MAX_TRAIN_BATCHES * BATCH_SIZE if MAX_TRAIN_BATCHES else "all",
)
train_settings = dict(
    batch_size=BATCH_SIZE,
    accumulation=ACCUMULATE_GRAD_BATCHES,
    max_epochs=MAX_EPOCHS,
    max_train_batches=MAX_TRAIN_BATCHES,
    hidden_size=HIDDEN_SIZE,
    lstm_layers=LSTM_LAYERS,
    heads=HEADS,
    dropout=DROPOUT,
    precision=PRECISION,
    csv_log_dir=CSV_LOGGER.log_dir,
    tensorboard_log_dir=TENSORBOARD_LOGGER.log_dir,
)
write_json(FIT_DIR / "training_settings.json", train_settings)
model = TFTModel(
    input_chunk_length=ICL,
    output_chunk_length=OCL,
    hidden_size=HIDDEN_SIZE,
    lstm_layers=LSTM_LAYERS,
    num_attention_heads=HEADS,
    dropout=DROPOUT,
    batch_size=BATCH_SIZE,
    n_epochs=MAX_EPOCHS,
    likelihood=NegativeBinomialLikelihood(),
    loss_fn=None,
    use_reversible_instance_norm=False,
    categorical_embedding_sizes=embeddings,
    random_state=42,
    add_relative_index=True,
    save_checkpoints=True,
    force_reset=False,
    model_name=MODEL_NAME,
    work_dir=str(WORK_DIR),
    pl_trainer_kwargs=dict(
        accelerator="gpu",
        devices=1,
        precision=PRECISION,
        callbacks=[
            EarlyStopping(monitor="val_loss", patience=5, min_delta=0.0001, mode="min")
        ],
        gradient_clip_val=0.1,
        accumulate_grad_batches=ACCUMULATE_GRAD_BATCHES,
        limit_train_batches=MAX_TRAIN_BATCHES if MAX_TRAIN_BATCHES else 1.0,
        limit_val_batches=1.0,
        logger=[CSV_LOGGER, TENSORBOARD_LOGGER],
        log_every_n_steps=50,
        enable_progress_bar=True,
    ),
)
model.fit(
    series=train_seq,
    future_covariates=train_cov,
    val_series=val_seq,
    val_future_covariates=val_cov,
    sample_weight=train_weights,
    val_sample_weight=val_weights,
    dataloader_kwargs={"num_workers": 0, "pin_memory": True},
    verbose=True,
)
if model.trainer.strategy.root_device.type != "cuda":
    raise RuntimeError("Training did not use CUDA")
embedding_count = sum((p.numel() for p in model.model.input_embeddings.parameters()))
if embedding_count <= 0:
    raise RuntimeError("Categorical embeddings were not built")
print("GPU training verified. Embedding parameters:", embedding_count)
best = TFTModel.load_from_checkpoint(
    model_name=MODEL_NAME, work_dir=str(WORK_DIR), best=True, map_location="cpu"
)
assert best.input_chunk_length == ICL and best.output_chunk_length == OCL
files = [
    "cache_manifest.json",
    "static_raw.parquet",
    "static_transformer.pkl",
    "static_encoded.pkl",
    "shared_cov.pkl",
    "training_settings.json",
]
checkpoint_files = list(WORK_DIR.rglob("*"))
files += [
    str(p.relative_to(FIT_DIR))
    for p in checkpoint_files
    if p.is_file() and (p.suffix == ".ckpt" or p.name.endswith(".pth.tar"))
]
bundle = dict(
    run_id=cfg["run_id"],
    model_name=MODEL_NAME,
    work_dir=str(WORK_DIR.resolve()),
    config_hash=dm["config_hash"],
    calendar_hash=dm["calendar_hash"],
    versions={
        "darts": darts.__version__,
        "torch": torch.__version__,
        "lightning": pl.__version__,
    },
    file_hashes={name: digest(FIT_DIR / name) for name in files},
    series_count=len(records),
    gpu=torch.cuda.get_device_name(0),
    embedding_parameters=embedding_count,
    csv_log_dir=CSV_LOGGER.log_dir,
    tensorboard_log_dir=TENSORBOARD_LOGGER.log_dir,
)
write_json(FIT_DIR / "bundle.json", bundle)
write_json(
    RUN_DIR / "active_fit.json",
    {"fit_dir": str(FIT_DIR.resolve()), "bundle_hash": digest(FIT_DIR / "bundle.json")},
)
print("Best checkpoint and matching artifacts ready:", FIT_DIR)
print("Next: prediction_code_iteration3_updated.ipynb")
